# 56 — Operational Readiness

Sintetiza evidencia M6 y genera runbooks operacionales. `shadow_ready` y `production_promotion_authorized` son estados distintos: M6 puede demostrar operación segura en shadow sin autorizar promoción automática.

In [1]:
import pandas as pd
from pathlib import Path

from argos.common.paths import ProjectPaths
from argos.config.gold import load_gold_spec
from argos.config.modeling import load_modeling_spec
from argos.config.production import load_production_spec

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()

m5_gate = paths.docs / "decisions" / "M5_GATE.md"
assert m5_gate.exists() and "M5 ✅ CLOSED" in m5_gate.read_text(encoding="utf-8"), (
    "M5 no está formalmente cerrado. No iniciar M6."
)

GOLD_SPEC = load_gold_spec(paths.configs / "gold.toml")
MODEL_SPEC = load_modeling_spec(paths.configs / "modeling.toml")
M6_SPEC = load_production_spec(paths.configs / "production.toml")
MANIFEST_DIR = paths.root / "artifacts" / "manifests"
TEMP_DIR = paths.root / ".cache" / "duckdb"
SHADOW_INPUT = paths.root / "data" / "m5" / "fresh_oot_2026.parquet"

assert GOLD_SPEC.schema_version == M6_SPEC.core.benchmark_gold_schema
assert MODEL_SPEC.schema_version == M6_SPEC.core.benchmark_model_schema

{
    "m6_schema": M6_SPEC.schema_version,
    "model_version": M6_SPEC.core.model_version,
    "operating_mode": M6_SPEC.core.operating_mode,
    "scoring_mode": M6_SPEC.core.scoring_mode,
    "human_review_required": M6_SPEC.core.human_review_required,
    "production_promotion_authorized": M6_SPEC.deployment.allow_production_promotion,
}


{'m6_schema': 'm6-production-v1',
 'model_version': 'argos-v1',
 'operating_mode': 'shadow',
 'scoring_mode': 'batch',
 'human_review_required': True,
 'production_promotion_authorized': False}

In [2]:
from argos.production.deployment_assets import ASSET_BOOTSTRAP_IMPLEMENTATION, ensure_deployment_assets
from argos.production.readiness import BUILD_IMPLEMENTATION, build_operational_readiness

print("asset bootstrap:", ASSET_BOOTSTRAP_IMPLEMENTATION)
print("readiness implementation:", BUILD_IMPLEMENTATION)
asset_status = ensure_deployment_assets(paths.root)
display(pd.DataFrame(asset_status))
assert all(item["exists"] for item in asset_status)

result = build_operational_readiness(
    root=paths.root, production_spec=M6_SPEC, reports_root=paths.reports, manifest_dir=MANIFEST_DIR
)
result.evidence


asset bootstrap: m6-deployment-assets-bootstrap-v1
readiness implementation: m6-56-operational-readiness-v3


,asset,target,template,action,exists,matches_template,sha256
0,dockerignore,.dockerignore,deploy/templates/dockerignore,created,True,True,858f7677d8e6957efec3bd7d3b6e81b50f5c57e5c1f51b...
1,github_ci,.github/workflows/ci.yml,deploy/templates/github_ci.yml,created,True,True,738375455a90c19e77f072db0b0b3ca869bfb60563c1ab...


,evidence,present,detail
0,components_50_55,True,"{'50': {'exists': True, 'blocking': 5, 'passed..."
1,container_contract,True,Dockerfile
2,dockerignore_contract,True,.dockerignore
3,ci_contract,True,.github/workflows/ci.yml
4,audit_chain,True,events=16; error=None
5,human_review,True,required
6,production_auto_promotion,True,disabled


In [3]:
result.validation

,check_id,category,blocking,passed,observed,requirement
0,m6_components_50_55_complete,evidence,True,True,"{""50"": {""blocking"": 5, ""exists"": true, ""ok"": t...",all M6 operational components before readiness...
1,deployment_assets_present,deployment,True,True,"{""missing"": []}","container, CI and operator entrypoints are ver..."
2,operational_docs_present,documentation,True,True,"{""paths"": [""/Users/macm1pro/Desktop/projects/a...","operations, incident, human review, retraining..."
3,audit_chain_operational,audit,True,True,"{""error"": null, ""events"": 16}",hash-chained audit logging contains end-to-end...
4,human_in_loop_is_enforced,governance,True,True,"{""human_review_required"": true, ""production_pr...","productionization automates prioritization, no..."
5,high_alerts_remain_visible,monitoring,True,True,"{""block_reason"": ""6 high-severity monitoring a...",deployment state explicitly carries monitoring...


In [4]:
blocking = result.validation.loc[result.validation["blocking"]]
failed = blocking.loc[~blocking["passed"], ["check_id", "category", "observed", "requirement"]]

print("56 ✅ PASS" if result.ready_for_next_notebook else "56 ❌ FAIL")
print(f"Blocking checks: {int(blocking['passed'].sum())}/{len(blocking)}")
if not failed.empty:
    print("\nBlocking failures:")
    display(failed)
print("Operational runbook:", result.runbook_path)
print("Incident response:", result.incident_path)
print("Human review SOP:", result.human_review_path)
print("Retraining policy:", result.retraining_path)
print("Security & access:", result.security_path)
print("Manifest:", result.manifest_path)
assert result.ready_for_next_notebook


56 ✅ PASS
Blocking checks: 6/6
Operational runbook: /Users/macm1pro/Desktop/projects/argos/argos-code/docs/operations/ARGOS_V1_OPERATIONAL_RUNBOOK.md
Incident response: /Users/macm1pro/Desktop/projects/argos/argos-code/docs/operations/INCIDENT_RESPONSE.md
Human review SOP: /Users/macm1pro/Desktop/projects/argos/argos-code/docs/operations/HUMAN_REVIEW_SOP.md
Retraining policy: /Users/macm1pro/Desktop/projects/argos/argos-code/docs/operations/RETRAINING_POLICY.md
Security & access: /Users/macm1pro/Desktop/projects/argos/argos-code/docs/operations/SECURITY_AND_ACCESS.md
Manifest: /Users/macm1pro/Desktop/projects/argos/argos-code/artifacts/manifests/m6_operational_readiness.json
